# Data Merge Pipeline

Combines 5 cleaned datasets into a single **city-level master CSV** used by the ML model and API.

**Join architecture**
```
uscities_latlon  <- spine (city + county keys)
    LEFT JOIN crime          on city_state_key   (city-level, sparse ~972 cities)
    LEFT JOIN nri_weather    on county_state_key  (county-level, all cities in same county share)
    LEFT JOIN zillow_agg     on county_state_key  (county-level, trailing 12-month avg rent)
    LEFT JOIN col_pivoted    on county_state_key  (county-level, 7 family-size profiles -> pivoted wide)
```

Multiple cities in the same county inherit the same county-level rent, weather, and cost-of-living data.
Cost of living has 7 rows per county (one per family profile) - these are pivoted into 49 columns so each county ends up as a single row.

In [ ]:
import pandas as pd
import numpy as np

# -- Load all cleaned datasets -------------------------------------------------
cities  = pd.read_csv('../cleaned_data/uscities_latlon_cleaned.csv')
crime   = pd.read_csv('../cleaned_data/crime_cleaned.csv')
weather = pd.read_csv('../cleaned_data/nri_weather_risk_cleaned.csv')
col     = pd.read_csv('../cleaned_data/cost_of_living_us_cleaned.csv')
zillow  = pd.read_csv('../cleaned_data/zillow_zori_rent_cleaned.csv')

print("Loaded datasets:")
print(f"  cities   : {cities.shape}")
print(f"  crime    : {crime.shape}")
print(f"  weather  : {weather.shape}")
print(f"  col      : {col.shape}  ← 7 rows per county (one per family profile)")
print(f"  zillow   : {zillow.shape}  ← 136 monthly rows per county")

In [ ]:
# -- Step 1: Aggregate Zillow → 1 row per county (trailing 12-month avg rent) --
#
# Zillow has 136 monthly rows per county (Jan 2015 – Apr 2026).
# Strategy: average the most recent 12 months so the rent figure reflects
# current market conditions and smooths seasonal swings.

cutoff = zillow['date'].max() - pd.DateOffset(months=12)
zillow_recent = zillow[zillow['date'] > cutoff]

zillow_agg = (
    zillow_recent
    .groupby('county_state_key')['zori']
    .mean()
    .reset_index()
    .rename(columns={'zori': 'avg_monthly_rent'})
)

zillow_agg['avg_monthly_rent'] = zillow_agg['avg_monthly_rent'].round(2)

print(f"Zillow aggregated: {zillow_agg.shape[0]} counties")
print(f"Date window used : {cutoff.date()} → {zillow['date'].max().date()}")
print(zillow_agg.head(5))

In [ ]:
# -- Step 2: Pivot Cost of Living wide → 1 row per county ---------------------
#
# Each county has 7 rows, one per family profile:
#   1p0c = 1 adult  0 children  |  2p0c = 2 adults 0 children
#   1p1c = 1 adult  1 child     |  2p1c = 2 adults 1 child
#   1p2c = 1 adult  2 children  |  2p2c = 2 adults 2 children
#   1p3c = 1 adult  3 children
#
# Pivot so each profile becomes a column prefix, e.g.:
#   1p0c_total_cost, 1p0c_food_cost, 2p2c_childcare_cost, ...
# Result: 1 row per county, 56 cost columns (7 profiles × 8 metrics)
# Plus median_family_income kept as a single county-level column.

cost_metrics = [
    'housing_cost',          # MIT living wage housing (kept for reference; API uses Zillow rent)
    'food_cost',
    'transportation_cost',
    'healthcare_cost',
    'other_necessities_cost',
    'childcare_cost',
    'taxes',
    'total_cost',
]

# Pivot: rows = county, columns = (metric, family_profile)
col_pivot = col.pivot_table(
    index='county_state_key',
    columns='family_member_count',
    values=cost_metrics,
    aggfunc='first'   # already 1 row per county per profile
)

# Flatten multi-index columns → "profile_metric" (e.g. 1p0c_food_cost)
col_pivot.columns = [f'{profile}_{metric}' for metric, profile in col_pivot.columns]
col_pivot = col_pivot.reset_index()

# Add median_family_income as a single county-level figure (same across all profiles)
col_income = (
    col.groupby('county_state_key')['median_family_income']
    .first()
    .reset_index()
)
col_pivot = col_pivot.merge(col_income, on='county_state_key', how='left')

print(f"CoL pivoted: {col_pivot.shape[0]} counties, {col_pivot.shape[1]} columns")
print("Sample columns:", col_pivot.columns[:10].tolist(), "...")
print(col_pivot.head(3))

In [ ]:
# -- Step 3: Build master city-level dataset ----------------------------------
#
# Spine: uscities_latlon - every city with both city_state_key and county_state_key.
# Multiple cities can share the same county → they inherit the same county-level data.
# All joins are LEFT so cities are never dropped for missing county/crime data.

master = cities.copy()

# 1) Crime - city level (sparse: ~972 out of 28k cities, rest will be NaN)
master = master.merge(
    crime[['city_state_key', 'population', 'violent_crime_rate', 'property_crime_rate']],
    on='city_state_key',
    how='left'
)

# 2) Weather risk - county level (all cities in same county share scores)
weather_cols = ['county_state_key', 'county_fips',
                'hurricane_risk_score', 'wildfire_risk_score',
                'tornado_risk_score', 'flood_risk_score']
master = master.merge(
    weather[weather_cols],
    on='county_state_key',
    how='left'
)

# 3) Rent - county level (trailing 12-month Zillow avg, one value per county)
master = master.merge(
    zillow_agg,
    on='county_state_key',
    how='left'
)

# 4) Cost of living - county level, pivoted wide (56 cost cols + median_family_income)
master = master.merge(
    col_pivot,
    on='county_state_key',
    how='left'
)

print(f"Master shape: {master.shape}")
print(f"Columns ({len(master.columns)}):")
print(master.columns.tolist())
print()
print(master.head(3))

In [ ]:
# -- Step 4: Validate coverage & save -----------------------------------------

total = len(master)

print("=" * 55)
print(f"  Master dataset: {total:,} cities")
print("=" * 55)

coverage = {
    'avg_monthly_rent (Zillow)'   : 'avg_monthly_rent',
    'violent_crime_rate (FBI)'    : 'violent_crime_rate',
    'hurricane_risk_score (FEMA)' : 'hurricane_risk_score',
    '1p0c_total_cost (MIT CoL)'   : '1p0c_total_cost',
    'lat/lng'                     : 'lat',
}

for label, col_name in coverage.items():
    count = master[col_name].notna().sum()
    pct   = count / total * 100
    print(f"  {label:<35} {count:>6,}  ({pct:.1f}%)")

print()

# Check for duplicate city rows (shouldn't happen)
dupes = master.duplicated(subset='city_state_key').sum()
print(f"  Duplicate city_state_key rows : {dupes}")

# Quick sanity: a city sharing a county should have the same rent
sample_county = master[master['county_state_key'].notna()].groupby('county_state_key').filter(lambda g: len(g) > 1)
if not sample_county.empty:
    example = sample_county['county_state_key'].iloc[0]
    print(f"\n  Example - cities in county '{example}':")
    print(sample_county[sample_county['county_state_key'] == example][
        ['city', 'county', 'avg_monthly_rent', 'violent_crime_rate']
    ].to_string(index=False))

print()

# Save
out_path = '../cleaned_data/master_data.csv'
master.to_csv(out_path, index=False)
print(f"  Saved → {out_path}")
print(f"  File columns : {master.shape[1]}")
print(f"  File rows    : {master.shape[0]:,}")

In [ ]:
import pandas as pd
import numpy as np

# -- Load all cleaned datasets -------------------------------------------------
cities  = pd.read_csv('../cleaned_data/uscities_latlon_cleaned.csv')
crime   = pd.read_csv('../cleaned_data/crime_cleaned.csv')
weather = pd.read_csv('../cleaned_data/nri_weather_risk_cleaned.csv')
col     = pd.read_csv('../cleaned_data/cost_of_living_us_cleaned.csv')
zillow  = pd.read_csv('../cleaned_data/zillow_zori_rent_cleaned.csv')

print("Loaded datasets:")
print(f"  cities   : {cities.shape}")
print(f"  crime    : {crime.shape}")
print(f"  weather  : {weather.shape}")
print(f"  col      : {col.shape}  ← 7 rows per county (one per family profile)")
print(f"  zillow   : {zillow.shape}  ← 136 monthly rows per county")

# -- Step 1: Aggregate Zillow → 1 row per county (trailing 12-month avg rent) --
#
# Zillow has 136 monthly rows per county (Jan 2015 – Apr 2026).
# Strategy: average the most recent 12 months so the rent figure reflects
# current market conditions and smooths seasonal swings.

zillow['date'] = pd.to_datetime(zillow['date'])
cutoff = zillow['date'].max() - pd.DateOffset(months=12)
zillow_recent = zillow[zillow['date'] > cutoff]

zillow_agg = (
    zillow_recent
    .groupby('county_state_key')['zori']
    .mean()
    .reset_index()
    .rename(columns={'zori': 'avg_monthly_rent'})
)

zillow_agg['avg_monthly_rent'] = zillow_agg['avg_monthly_rent'].round(2)

print(f"\nZillow aggregated: {zillow_agg.shape[0]} counties")
print(f"Date window used : {cutoff.date()} → {zillow['date'].max().date()}")
print(zillow_agg.head(5))

# -- Step 2: Pivot Cost of Living wide → 1 row per county ---------------------
#
# Each county has 7 rows, one per family profile:
#   1p0c = 1 adult  0 children  |  2p0c = 2 adults 0 children
#   1p1c = 1 adult  1 child     |  2p1c = 2 adults 1 child
#   1p2c = 1 adult  2 children  |  2p2c = 2 adults 2 children
#   1p3c = 1 adult  3 children
#
# Pivot so each profile becomes a column prefix, e.g.:
#   1p0c_total_cost, 1p0c_food_cost, 2p2c_childcare_cost, ...
# Result: 1 row per county, 56 cost columns (7 profiles × 8 metrics)
# Plus median_family_income kept as a single county-level column.

cost_metrics = [
    'housing_cost',          # MIT living wage housing (kept for reference; API uses Zillow rent)
    'food_cost',
    'transportation_cost',
    'healthcare_cost',
    'other_necessities_cost',
    'childcare_cost',
    'taxes',
    'total_cost',
]

# Pivot: rows = county, columns = (metric, family_profile)
col_pivot = col.pivot_table(
    index='county_state_key',
    columns='family_member_count',
    values=cost_metrics,
    aggfunc='first'   # already 1 row per county per profile
)

# Flatten multi-index columns → "profile_metric" (e.g. 1p0c_food_cost)
col_pivot.columns = [f'{profile}_{metric}' for metric, profile in col_pivot.columns]
col_pivot = col_pivot.reset_index()

# Add median_family_income as a single county-level figure (same across all profiles)
col_income = (
    col.groupby('county_state_key')['median_family_income']
    .first()
    .reset_index()
)
col_pivot = col_pivot.merge(col_income, on='county_state_key', how='left')

print(f"\nCoL pivoted: {col_pivot.shape[0]} counties, {col_pivot.shape[1]} columns")
print("Sample columns:", col_pivot.columns[:10].tolist(), "...")
print(col_pivot.head(3))

# -- Step 3: Build master city-level dataset ----------------------------------
#
# Spine: uscities_latlon - every city with both city_state_key and county_state_key.
# Multiple cities can share the same county → they inherit the same county-level data.
# All joins are LEFT so cities are never dropped for missing county/crime data.

master = cities.copy()

# 1) Crime - city level (sparse: ~972 out of 28k cities, rest will be NaN)
master = master.merge(
    crime[['city_state_key', 'population', 'violent_crime_rate', 'property_crime_rate']],
    on='city_state_key',
    how='left'
)

# 2) Weather risk - county level (all cities in same county share scores)
weather_cols = ['county_state_key', 'county_fips',
                'hurricane_risk_score', 'wildfire_risk_score',
                'tornado_risk_score', 'flood_risk_score']
master = master.merge(
    weather[weather_cols],
    on='county_state_key',
    how='left'
)

# 3) Rent - county level (trailing 12-month Zillow avg, one value per county)
#    avg_monthly_rent comes from zillow_agg built in Step 1
master = master.merge(
    zillow_agg[['county_state_key', 'avg_monthly_rent']],
    on='county_state_key',
    how='left'
)

# 4) Cost of living - county level, pivoted wide (56 cost cols + median_family_income)
master = master.merge(
    col_pivot,
    on='county_state_key',
    how='left'
)

print(f"\nMaster shape: {master.shape}")
print(f"Columns ({len(master.columns)}):")
print(master.columns.tolist())
print()
print(master.head(3))

# -- Step 4: Validate coverage & save -----------------------------------------

total = len(master)

print("=" * 55)
print(f"  Master dataset: {total:,} cities")
print("=" * 55)

coverage = {
    'avg_monthly_rent (Zillow)'   : 'avg_monthly_rent',
    'violent_crime_rate (FBI)'    : 'violent_crime_rate',
    'hurricane_risk_score (FEMA)' : 'hurricane_risk_score',
    '1p0c_total_cost (MIT CoL)'   : '1p0c_total_cost',
    'lat/lng'                     : 'lat',
}

for label, col_name in coverage.items():
    count = master[col_name].notna().sum()
    pct   = count / total * 100
    print(f"  {label:<35} {count:>6,}  ({pct:.1f}%)")

print()

# Check for duplicate city rows (shouldn't happen)
dupes = master.duplicated(subset='city_state_key').sum()
print(f"  Duplicate city_state_key rows : {dupes}")

# Quick sanity: a city sharing a county should have the same rent
sample_county = master[master['county_state_key'].notna()].groupby('county_state_key').filter(lambda g: len(g) > 1)
if not sample_county.empty:
    example = sample_county['county_state_key'].iloc[0]
    print(f"\n  Example - cities in county '{example}':")
    print(sample_county[sample_county['county_state_key'] == example][
        ['city', 'county', 'avg_monthly_rent', 'violent_crime_rate']
    ].to_string(index=False))

print()

# Save
out_path = '../cleaned_data/master_data.csv'
master.to_csv(out_path, index=False)
print(f"  Saved → {out_path}")
print(f"  File columns : {master.shape[1]}")
print(f"  File rows    : {master.shape[0]:,}")
